In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="J1mmAuxLk77g4eiOvCmr")
project = rf.workspace("toicheckdataset").project("teting_generade_weapon")
version = project.version(1)
dataset = version.download("yolov8")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 36.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 107.5 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: idna
    Found existing installation: idna 3.10
    Uninstalling idna-3.10:
      Successfully uninstalled idna-3.10
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.11.0.86
    Uninstalling opencv-python-headless-4.11.0.86:
      Successfully uninstalled opencv-python-headless-4.11.0.86
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sigstore 3.6.1 requires rich~=13.0, but you have rich 14.0.0 which is incompatible.
datasets 3.5.0 requires fsspec[http]<=


Extracting Dataset Version Zip to teting_generade_weapon-1 in yolov8:: 100%|██████████| 4397/4397 [00:00<00:00, 7838.68it/s]


In [3]:
import os
os.listdir('/kaggle/working/teting_generade_weapon-1/')

['train', 'README.roboflow.txt', 'data.yaml', 'test', 'README.dataset.txt']

In [4]:
!pip install torch torchvision pycocotools albumentations
!pip install git+https://github.com/rwightman/efficientdet-pytorch.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 82.8 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.8.93
    Uninstalling nvidia-nvjitlink-cu12-12.8.93:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.8.93
  Attempting uninstall: nvidia-curand-cu12
    Found existing installation: nvidia-curand-cu12 10.3.9.90
    Uninstalling nvidia-curand-cu12-1

In [5]:

import os
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import cv2
from sklearn.metrics import precision_recall_curve, average_precision_score, precision_score, recall_score, f1_score

# Let's examine the dataset structure
base_path = '/kaggle/working/teting_generade_weapon-1/'
print("Files in the main directory:", os.listdir(base_path))

# Read the YAML file to understand the dataset configuration
with open(os.path.join(base_path, 'data.yaml'), 'r') as yaml_file:
    data_config = yaml.safe_load(yaml_file)
    print("Dataset configuration:")
    print(data_config)

# Check the train and test directories
print("\nTraining data:", os.listdir(os.path.join(base_path, 'train')))
print("Test data:", os.listdir(os.path.join(base_path, 'test')))

# Check if the images and labels directories exist in train and test
train_images_path = os.path.join(base_path, 'train', 'images')
train_labels_path = os.path.join(base_path, 'train', 'labels')
test_images_path = os.path.join(base_path, 'test', 'images')
test_labels_path = os.path.join(base_path, 'test', 'labels')

# Verify images and labels
if os.path.exists(train_images_path) and os.path.exists(train_labels_path):
    print(f"\nTrain images count: {len(os.listdir(train_images_path))}")
    print(f"Train labels count: {len(os.listdir(train_labels_path))}")
else:
    print("\nTrain images or labels directory not found.")

if os.path.exists(test_images_path) and os.path.exists(test_labels_path):
    print(f"Test images count: {len(os.listdir(test_images_path))}")
    print(f"Test labels count: {len(os.listdir(test_labels_path))}")
else:
    print("Test images or labels directory not found.")

# Let's examine a sample label file to understand the format
try:
    sample_label_files = os.listdir(train_labels_path)
    if sample_label_files:
        with open(os.path.join(train_labels_path, sample_label_files[0]), 'r') as label_file:
            print("\nSample label file content:")
            print(label_file.read())
except Exception as e:
    print(f"Error reading sample label: {e}")

2025-05-15 03:33:05.121638: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747279985.308721      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747279985.361183      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Files in the main directory: ['train', 'README.roboflow.txt', 'data.yaml', 'test', 'README.dataset.txt']
Dataset configuration:
{'names': ['Grenade', 'Gun', 'Handgun', 'Knife', 'Person', 'Weapon Holding'], 'nc': 6, 'roboflow': {'license': 'CC BY 4.0', 'project': 'teting_generade_weapon', 'url': 'https://universe.roboflow.com/toicheckdataset/teting_generade_weapon/dataset/1', 'version': 1, 'workspace': 'toicheckdataset'}, 'test': '../test/images', 'train': '../train/images', 'val': '../valid/images'}

Training data: ['labels', 'images']
Test data: ['labels', 'images']

Train images count: 1938
Train labels count: 1938
Test images count: 256
Test labels count: 256

Sample label file content:
0 0.35466308231457433 0.48231721671852174 0.07242827792830413 0.1011727621748646
1 0.24562775553860855 0.79765625 0.4288574988683463 0.4046875
4 0.5847976396415205 0.5019773315738997 0.7164702792830411 0.9960453368522006


In [1]:

import os
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from tqdm.auto import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from effdet import get_efficientdet_config, EfficientDet, DetBenchTrain, DetBenchPredict
from effdet.efficientdet import HeadNet
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import precision_recall_curve, average_precision_score, precision_score, recall_score, f1_score

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Path configurations
base_path = '/kaggle/working/teting_generade_weapon-1/'
train_img_dir = os.path.join(base_path, 'train/images')
train_label_dir = os.path.join(base_path, 'train/labels')
test_img_dir = os.path.join(base_path, 'test/images')
test_label_dir = os.path.join(base_path, 'test/labels')

# Load class names from YAML
with open(os.path.join(base_path, 'data.yaml'), 'r') as yaml_file:
    data_config = yaml.safe_load(yaml_file)
    class_names = data_config['names']
    num_classes = data_config['nc']
    
print(f"Classes: {class_names}")
print(f"Number of classes: {num_classes}")

# Convert YOLO format to EfficientDet format
def yolo_to_efficientdet(yolo_label, img_width, img_height):
    """
    Convert YOLO format (class_id, x_center, y_center, width, height) to 
    EfficientDet format (x_min, y_min, x_max, y_max, class_id)
    """
    bboxes = []
    labels = []
    
    with open(yolo_label, 'r') as f:
        for line in f:
            data = line.strip().split(' ')
            if len(data) < 5:  # Skip malformed lines
                continue
                
            try:
                class_id = int(data[0])
                x_center = float(data[1])
                y_center = float(data[2])
                width = float(data[3])
                height = float(data[4])
                
                # Convert to absolute coordinates
                x_min = max(0, (x_center - width/2) * img_width)
                y_min = max(0, (y_center - height/2) * img_height)
                x_max = min(img_width, (x_center + width/2) * img_width)
                y_max = min(img_height, (y_center + height/2) * img_height)
                
                # Skip invalid boxes (may happen due to rounding errors)
                if x_max <= x_min or y_max <= y_min:
                    continue
                    
                bboxes.append([x_min, y_min, x_max, y_max])
                labels.append(class_id)
            except (ValueError, IndexError):
                continue
    
    return np.array(bboxes, dtype=np.float32), np.array(labels, dtype=np.int64)

class WeaponDetectionDataset(Dataset):
    def __init__(self, img_dir, label_dir, transforms=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transforms = transforms
        self.img_files = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
        
    def __len__(self):
        return len(self.img_files)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_files[idx])
        img = cv2.imread(img_path)
        if img is None:
            print(f"Warning: Could not read image {img_path}")
            # Create a dummy image
            img = np.zeros((512, 512, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Get image dimensions
        height, width = img.shape[:2]
        
        # Get label path
        label_path = os.path.join(self.label_dir, self.img_files[idx].replace('.jpg', '.txt').replace('.png', '.txt').replace('.jpeg', '.txt'))
        
        # Load boxes and labels
        if os.path.exists(label_path):
            boxes, labels = yolo_to_efficientdet(label_path, width, height)
            # If no valid boxes, set default
            if len(boxes) == 0:
                boxes = np.zeros((0, 4), dtype=np.float32)
                labels = np.zeros(0, dtype=np.int64)
        else:
            print(f"Warning: No label file found for {img_path}")
            boxes = np.zeros((0, 4), dtype=np.float32)
            labels = np.zeros(0, dtype=np.int64)
        
        # Apply transformations
        if self.transforms:
            # Check if boxes are empty
            if len(boxes) == 0:
                # Just transform the image
                transformed = self.transforms(image=img)
                img = transformed['image']
            else:
                try:
                    transformed = self.transforms(image=img, bboxes=boxes, labels=labels)
                    img = transformed['image']
                    boxes = np.array(transformed['bboxes'], dtype=np.float32)
                    labels = np.array(transformed['labels'], dtype=np.int64)
                except Exception as e:
                    print(f"Error in transformation: {e} for image {img_path}")
                    boxes = np.zeros((0, 4), dtype=np.float32)
                    labels = np.zeros(0, dtype=np.int64)
                    transformed = self.transforms(image=img)
                    img = transformed['image']
        
        # Convert to tensor if not already
        if not isinstance(img, torch.Tensor):
            img = torch.from_numpy(img).permute(2, 0, 1).float()
        
        boxes = torch.from_numpy(boxes).float()
        labels = torch.from_numpy(labels).long()
        
        return {
            'image': img,
            'boxes': boxes,
            'labels': labels,
            'img_path': img_path,
            'image_id': idx
        }

# Define transformations
def get_train_transforms(target_size=(512, 512)):
    return A.Compose([
        A.Resize(height=target_size[0], width=target_size[1]),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(p=1)
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

def get_valid_transforms(target_size=(512, 512)):
    return A.Compose([
        A.Resize(height=target_size[0], width=target_size[1]),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(p=1)
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

# Collate function for batch processing
def collate_fn(batch):
    images = [item['image'] for item in batch]
    boxes = [item['boxes'] for item in batch]
    labels = [item['labels'] for item in batch]
    img_paths = [item['img_path'] for item in batch]
    image_ids = [item['image_id'] for item in batch]
    
    # Stack images
    images = torch.stack(images)
    
    return {
        'images': images,
        'boxes': boxes,
        'labels': labels,
        'img_paths': img_paths,
        'image_ids': image_ids
    }

# Initialize the EfficientDet model
def create_model(num_classes, image_size=512, architecture='efficientdet_d0'):
    config = get_efficientdet_config(architecture)
    config.image_size = (image_size, image_size)
    config.norm_kwargs = dict(eps=0.001, momentum=0.01)
    
    # Update the config with the number of classes
    config.num_classes = num_classes
    
    # Create the model
    backbone_net = EfficientDet(config, pretrained_backbone=True)
    
    # Create custom head
    head_net = HeadNet(config, num_outputs=num_classes)
    
    # Create model
    model = DetBenchTrain(backbone_net, head_net)
    
    return model

# Training function - FIXED
def train_model(model, train_loader, optimizer, scheduler, num_epochs=30):
    model.train()
    losses = []
    
    for epoch in range(num_epochs):
        epoch_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        
        for batch in progress_bar:
            images = batch['images'].to(device)
            
            # Prepare target dictionary for EfficientDet
            batch_size = images.shape[0]
            all_boxes = []
            all_labels = []
            
            for i in range(batch_size):
                boxes = batch['boxes'][i].to(device)
                labels = batch['labels'][i].to(device)
                
                all_boxes.append(boxes)
                all_labels.append(labels)
            
            target = {
                'bbox': all_boxes,
                'cls': all_labels
            }
            
            # Forward pass
            loss_dict = model(images, target)  # Expect dictionary with loss components
            loss = loss_dict['loss']  # Extract the total loss
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # Update progress bar
            epoch_loss += loss.item()
            progress_bar.set_postfix(loss=loss.item())
        
        # Update learning rate
        scheduler.step()
        
        avg_loss = epoch_loss / len(train_loader)
        losses.append(avg_loss)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")
    
    return losses

# Inference function
def predict(model, test_loader, confidence_threshold=0.25):
    model.eval()
    all_predictions = []
    all_ground_truths = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Predicting"):
            images = batch['images'].to(device)
            img_paths = batch['img_paths']
            
            # Get predictions
            detections = model(images)
            
            # Process each image in the batch
            for i, (img_path, detection) in enumerate(zip(img_paths, detections)):
                # Ensure detection is on CPU
                detection = detection.cpu().numpy()
                
                # Extract boxes, scores, labels
                boxes = detection[:, :4]  # x1, y1, x2, y2
                scores = detection[:, 4]
                labels = detection[:, 5].astype(int)
                
                # Filter by confidence threshold
                mask = scores >= confidence_threshold
                boxes = boxes[mask]
                scores = scores[mask]
                labels = labels[mask]
                
                # Ground truth
                gt_boxes = batch['boxes'][i].cpu().numpy()
                gt_labels = batch['labels'][i].cpu().numpy()
                
                all_predictions.append({
                    'img_path': img_path,
                    'boxes': boxes,
                    'scores': scores,
                    'labels': labels
                })
                
                all_ground_truths.append({
                    'img_path': img_path,
                    'boxes': gt_boxes,
                    'labels': gt_labels
                })
    
    return all_predictions, all_ground_truths

# Visualization function
def visualize_predictions(predictions, ground_truths, class_names, num_samples=5, save_dir='output_predictions'):
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    # Select random samples
    indices = np.random.choice(len(predictions), min(num_samples, len(predictions)), replace=False)
    
    for idx in indices:
        pred = predictions[idx]
        gt = ground_truths[idx]
        
        img_path = pred['img_path']
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Create a figure with two subplots
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
        
        # Plot ground truth
        ax1.imshow(img)
        ax1.set_title('Ground Truth')
        
        for box, label in zip(gt['boxes'], gt['labels']):
            x1, y1, x2, y2 = box
            class_name = class_names[label]
            color = (0, 1, 0)  # Green for ground truth
            rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor=color, linewidth=2)
            ax1.add_patch(rect)
            ax1.text(x1, y1, f"{class_name}", bbox=dict(facecolor=color, alpha=0.5))
        
        # Plot predictions
        ax2.imshow(img)
        ax2.set_title('Predictions')
        
        for box, label, score in zip(pred['boxes'], pred['labels'], pred['scores']):
            x1, y1, x2, y2 = box
            class_name = class_names[label]
            color = (1, 0, 0)  # Red for predictions
            rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor=color, linewidth=2)
            ax2.add_patch(rect)
            ax2.text(x1, y1, f"{class_name}: {score:.2f}", bbox=dict(facecolor=color, alpha=0.5))
        
        # Save the figure
        plt.savefig(os.path.join(save_dir, f"pred_{os.path.basename(img_path)}"))
        plt.close()

# Calculate metrics
def calculate_metrics(predictions, ground_truths, iou_threshold=0.5):
    """
    Calculate precision, recall, and F1 score for object detection
    """
    all_true_positives = 0
    all_false_positives = 0
    all_false_negatives = 0
    
    for pred, gt in zip(predictions, ground_truths):
        true_positives = 0
        false_positives = 0
        false_negatives = 0
        
        # Match predictions to ground truth
        gt_matched = [False] * len(gt['boxes'])
        
        for pred_box, pred_label, pred_score in zip(pred['boxes'], pred['labels'], pred['scores']):
            matched = False
            
            for gt_idx, (gt_box, gt_label) in enumerate(zip(gt['boxes'], gt['labels'])):
                if gt_matched[gt_idx]:
                    continue
                
                # Calculate IoU
                x1 = max(pred_box[0], gt_box[0])
                y1 = max(pred_box[1], gt_box[1])
                x2 = min(pred_box[2], gt_box[2])
                y2 = min(pred_box[3], gt_box[3])
                
                # Calculate intersection area
                intersection_area = max(0, x2 - x1) * max(0, y2 - y1)
                
                # Calculate union area
                pred_area = (pred_box[2] - pred_box[0]) * (pred_box[3] - pred_box[1])
                gt_area = (gt_box[2] - gt_box[0]) * (gt_box[3] - gt_box[1])
                union_area = pred_area + gt_area - intersection_area
                
                # Calculate IoU
                iou = intersection_area / union_area if union_area > 0 else 0
                
                # Check if IoU is above threshold and labels match
                if iou >= iou_threshold and pred_label == gt_label:
                    true_positives += 1
                    gt_matched[gt_idx] = True
                    matched = True
                    break
            
            if not matched:
                false_positives += 1
        
        # Count ground truth that weren't matched
        false_negatives += sum(1 for matched in gt_matched if not matched)
        
        all_true_positives += true_positives
        all_false_positives += false_positives
        all_false_negatives += false_negatives
    
    # Calculate metrics
    precision = all_true_positives / (all_true_positives + all_false_positives) if (all_true_positives + all_false_positives) > 0 else 0
    recall = all_true_positives / (all_true_positives + all_false_negatives) if (all_true_positives + all_false_negatives) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'true_positives': all_true_positives,
        'false_positives': all_false_positives,
        'false_negatives': all_false_negatives
    }

# Main execution
if __name__ == "__main__":
    # Image size
    IMG_SIZE = 512
    
    # Hyperparameters
    BATCH_SIZE = 4
    EPOCHS = 50
    LEARNING_RATE = 0.0001
    
    # Create datasets
    train_dataset = WeaponDetectionDataset(
        img_dir=train_img_dir,
        label_dir=train_label_dir,
        transforms=get_train_transforms(target_size=(IMG_SIZE, IMG_SIZE))
    )
    
    test_dataset = WeaponDetectionDataset(
        img_dir=test_img_dir,
        label_dir=test_label_dir,
        transforms=get_valid_transforms(target_size=(IMG_SIZE, IMG_SIZE))
    )
    
    # Create data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=2
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=2
    )
    
    # Create model
    model = create_model(num_classes=num_classes, image_size=IMG_SIZE, architecture='efficientdet_d0')
    model.to(device)
    
    # Optimizer and scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
    
    # Train the model
    print("Training the model...")
    losses = train_model(model, train_loader, optimizer, scheduler, num_epochs=EPOCHS)
    
    # Plot loss curve
    plt.figure(figsize=(10, 5))
    plt.plot(losses)
    plt.title('Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.savefig('training_loss.png')
    plt.close()
    
    # Save the model
    model_save_path = 'efficientdet_weapon_detection.pth'
    torch.save({
        'model_state_dict': model.model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'epochs': EPOCHS,
        'num_classes': num_classes,
        'class_names': class_names
    }, model_save_path)
    print(f"Model saved to {model_save_path}")
    
    # Convert to inference mode for predictions
    bench_pred = DetBenchPredict(model.model)
    bench_pred.eval()
    bench_pred.to(device)
    
    # Make predictions
    print("Making predictions on test set...")
    predictions, ground_truths = predict(bench_pred, test_loader)
    
    # Visualize predictions
    print("Visualizing predictions...")
    visualize_predictions(predictions, ground_truths, class_names, num_samples=10)
    
    # Calculate metrics
    print("Calculating metrics...")
    metrics = calculate_metrics(predictions, ground_truths)
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall: {metrics['recall']:.4f}")
    print(f"F1 Score: {metrics['f1']:.4f}")
    print(f"True Positives: {metrics['true_positives']}")
    print(f"False Positives: {metrics['false_positives']}")
    print(f"False Negatives: {metrics['false_negatives']}")
    
    # Calculate class-wise metrics
    class_metrics = {}
    for class_id, class_name in enumerate(class_names):
        class_predictions = []
        class_ground_truths = []
        
        for pred, gt in zip(predictions, ground_truths):
            # Filter predictions by class
            class_pred_indices = np.where(pred['labels'] == class_id)[0]
            class_pred = {
                'img_path': pred['img_path'],
                'boxes': pred['boxes'][class_pred_indices] if len(class_pred_indices) > 0 else np.array([]),
                'scores': pred['scores'][class_pred_indices] if len(class_pred_indices) > 0 else np.array([]),
                'labels': pred['labels'][class_pred_indices] if len(class_pred_indices) > 0 else np.array([])
            }
            
            # Filter ground truths by class
            class_gt_indices = np.where(gt['labels'] == class_id)[0]
            class_gt = {
                'img_path': gt['img_path'],
                'boxes': gt['boxes'][class_gt_indices] if len(class_gt_indices) > 0 else np.array([]),
                'labels': gt['labels'][class_gt_indices] if len(class_gt_indices) > 0 else np.array([])
            }
            
            class_predictions.append(class_pred)
            class_ground_truths.append(class_gt)
        
        # Calculate metrics for this class
        class_metrics[class_name] = calculate_metrics(class_predictions, class_ground_truths)
        print(f"\nMetrics for class {class_name}:")
        print(f"Precision: {class_metrics[class_name]['precision']:.4f}")
        print(f"Recall: {class_metrics[class_name]['recall']:.4f}")
        print(f"F1 Score: {class_metrics[class_name]['f1']:.4f}")
    
    print("\nModel training and evaluation complete!")


/usr/local/lib/python3.11/dist-packages/albumentations/__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.6' (you have '2.0.4'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


Using device: cuda
Classes: ['Grenade', 'Gun', 'Handgun', 'Knife', 'Person', 'Weapon Holding']
Number of classes: 6
Training the model...


Epoch 1/50:   0%|          | 0/485 [00:00<?, ?it/s]

Epoch 1/50, Loss: 2.0693


Epoch 2/50:   0%|          | 0/485 [00:00<?, ?it/s]

...

Epoch 49/50, Loss: 1.0432


Epoch 50/50:   0%|          | 0/485 [00:00<?, ?it/s]

Epoch 50/50, Loss: 1.0459
Model saved to efficientdet_weapon_detection.pth
Making predictions on test set...


Predicting:   0%|          | 0/64 [00:00<?, ?it/s]

Visualizing predictions...
Calculating metrics...
Precision: 0.0400
Recall: 0.0315
F1 Score: 0.0352
True Positives: 33
False Positives: 792
False Negatives: 1016

Metrics for class Grenade:
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000

Metrics for class Gun:
Precision: 0.0772
Recall: 0.1150
F1 Score: 0.0924

Metrics for class Handgun:
Precision: 0.0421
Recall: 0.0517
F1 Score: 0.0464

Metrics for class Knife:
Precision: 0.0061
Recall: 0.0054
F1 Score: 0.0057

Metrics for class Person:
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000

Metrics for class Weapon Holding:
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000

Model training and evaluation complete!


In [2]:

import os
import yaml
import numpy as np
import cv2
import torch
from effdet import get_efficientdet_config, EfficientDet, DetBenchPredict
from effdet.efficientdet import HeadNet
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Path configurations
base_path = '/kaggle/working/teting_generade_weapon-1/'
model_path = '//kaggle/working/efficientdet_weapon_detection.pth'
data_yaml_path = os.path.join(base_path, 'data.yaml')
train_img_dir = os.path.join(base_path, 'train/images')
test_img_dir = os.path.join(base_path, 'test/images')

# Load class names from YAML
with open(data_yaml_path, 'r') as yaml_file:
    data_config = yaml.safe_load(yaml_file)
    class_names = data_config['names']
    num_classes = data_config['nc']

print(f"Classes: {class_names}")
print(f"Number of classes: {num_classes}")

# Define validation transforms
def get_valid_transforms(target_size=(512, 512)):
    return A.Compose([
        A.Resize(height=target_size[0], width=target_size[1]),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(p=1)
    ])

# Initialize the EfficientDet model
def create_model(num_classes, image_size=512, architecture='efficientdet_d0'):
    config = get_efficientdet_config(architecture)
    config.image_size = (image_size, image_size)
    config.norm_kwargs = dict(eps=0.001, momentum=0.01)
    config.num_classes = num_classes
    
    backbone_net = EfficientDet(config, pretrained_backbone=False)
    head_net = HeadNet(config, num_outputs=num_classes)
    model = DetBenchPredict(backbone_net)
    return model

# Suggest valid image files
def suggest_image_files(directory):
    valid_extensions = ('.jpg', '.jpeg', '.png')
    files = [f for f in os.listdir(directory) if f.lower().endswith(valid_extensions)]
    return files[:5]  # Return up to 5 valid files

# Predict on a single image and save output
def predict_single_image(model, image_path, output_path, confidence_threshold=0.25, image_size=512):
    model.eval()
    
    # Validate image path
    if not os.path.exists(image_path):
        raise FileNotFoundError(
            f"Image not found at {image_path}. "
            f"Check path or try one of these from train: {suggest_image_files(train_img_dir)} "
            f"or test: {suggest_image_files(test_img_dir)}"
        )
    
    # Read and preprocess image
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not read image at {image_path}. Ensure it's a valid image file.")
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Apply transformations
    transforms = get_valid_transforms(target_size=(image_size, image_size))
    transformed = transforms(image=img_rgb)
    img_tensor = transformed['image'].unsqueeze(0).to(device)
    
    # Run inference
    with torch.no_grad():
        detections = model(img_tensor)
    
    # Process detections
    detection = detections[0].cpu().numpy()
    boxes = detection[:, :4]  # x1, y1, x2, y2
    scores = detection[:, 4]
    labels = detection[:, 5].astype(int)
    
    # Filter by confidence threshold
    mask = scores >= confidence_threshold
    boxes = boxes[mask]
    scores = scores[mask]
    labels = labels[mask]
    
    # Draw predictions on the image
    img_annotated = img.copy()
    for box, score, label in zip(boxes, scores, labels):
        x1, y1, x2, y2 = box.astype(int)
        class_name = class_names[label]
        color = (0, 0, 255)  # Red (BGR)
        thickness = 2
        cv2.rectangle(img_annotated, (x1, y1), (x2, y2), color, thickness)
        label_text = f"{class_name}: {score:.2f}"
        cv2.putText(img_annotated, label_text, (x1, y1-10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, thickness)
    
    # Ensure output directory exists
    os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
    
    # Save annotated image
    cv2.imwrite(output_path, img_annotated)
    print(f"Annotated image saved to {output_path}")
    
    return boxes, scores, labels

# Main execution
if __name__ == "__main__":
    # Image size
    IMG_SIZE = 512
    
    # Input and output paths
    input_image_path = "/kaggle/working/teting_generade_weapon-1/train/images/00e8fdd668abf79ccc314a9121a7cfa5_jpg.rf.3d21243c06100c10e65f646fee015a5f.jpg"  # Corrected extension
    output_image_path = "/kaggle/working/annotated_output.jpg"
    
    # Create and load model
    model = create_model(num_classes=num_classes, image_size=IMG_SIZE, architecture='efficientdet_d0')
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model checkpoint not found at {model_path}")
    checkpoint = torch.load(model_path, map_location=device, weights_only=True)
    model.model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    print(f"Loaded model from {model_path}")
    
    # Predict and save
    try:
        boxes, scores, labels = predict_single_image(
            model, 
            input_image_path, 
            output_image_path, 
            confidence_threshold=0.25, 
            image_size=IMG_SIZE
        )
        print("Prediction complete!")
        print(f"Detected {len(boxes)} objects:")
        for box, score, label in zip(boxes, scores, labels):
            print(f"Class: {class_names[label]}, Score: {score:.2f}, Box: {box}")
    except Exception as e:
        print(f"Error during prediction: {e}")


Using device: cuda
Classes: ['Grenade', 'Gun', 'Handgun', 'Knife', 'Person', 'Weapon Holding']
Number of classes: 6
Loaded model from //kaggle/working/efficientdet_weapon_detection.pth
Annotated image saved to /kaggle/working/annotated_output.jpg
Prediction complete!
Detected 2 objects:
Class: Gun, Score: 0.31, Box: [  3.061554  14.845322 431.61948  443.2337  ]
Class: Handgun, Score: 0.27, Box: [  3.061554  14.845322 431.61948  443.2337  ]


In [3]:

import os
import yaml
import numpy as np
import cv2
import torch
from effdet import get_efficientdet_config, EfficientDet, DetBenchPredict
from effdet.efficientdet import HeadNet
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Path configurations
base_path = '/kaggle/working/teting_generade_weapon-1/'
model_path = '//kaggle/working/efficientdet_weapon_detection.pth'
data_yaml_path = os.path.join(base_path, 'data.yaml')
train_img_dir = os.path.join(base_path, 'train/images')
test_img_dir = os.path.join(base_path, 'test/images')

# Load class names from YAML
with open(data_yaml_path, 'r') as yaml_file:
    data_config = yaml.safe_load(yaml_file)
    class_names = data_config['names']
    num_classes = data_config['nc']

print(f"Classes: {class_names}")
print(f"Number of classes: {num_classes}")

# Define validation transforms
def get_valid_transforms(target_size=(512, 512)):
    return A.Compose([
        A.Resize(height=target_size[0], width=target_size[1]),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(p=1)
    ])

# Initialize the EfficientDet model
def create_model(num_classes, image_size=512, architecture='efficientdet_d0'):
    config = get_efficientdet_config(architecture)
    config.image_size = (image_size, image_size)
    config.norm_kwargs = dict(eps=0.001, momentum=0.01)
    config.num_classes = num_classes
    
    backbone_net = EfficientDet(config, pretrained_backbone=False)
    head_net = HeadNet(config, num_outputs=num_classes)
    model = DetBenchPredict(backbone_net)
    return model

# Suggest valid image files
def suggest_image_files(directory):
    valid_extensions = ('.jpg', '.jpeg', '.png')
    files = [f for f in os.listdir(directory) if f.lower().endswith(valid_extensions)]
    return files[:5]  # Return up to 5 valid files

# Predict on a single image and save output
def predict_single_image(model, image_path, output_path, confidence_threshold=0.25, image_size=512):
    model.eval()
    
    # Validate image path
    if not os.path.exists(image_path):
        raise FileNotFoundError(
            f"Image not found at {image_path}. "
            f"Check path or try one of these from train: {suggest_image_files(train_img_dir)} "
            f"or test: {suggest_image_files(test_img_dir)}"
        )
    
    # Read and preprocess image
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not read image at {image_path}. Ensure it's a valid image file.")
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Apply transformations
    transforms = get_valid_transforms(target_size=(image_size, image_size))
    transformed = transforms(image=img_rgb)
    img_tensor = transformed['image'].unsqueeze(0).to(device)
    
    # Run inference
    with torch.no_grad():
        detections = model(img_tensor)
    
    # Process detections
    detection = detections[0].cpu().numpy()
    boxes = detection[:, :4]  # x1, y1, x2, y2
    scores = detection[:, 4]
    labels = detection[:, 5].astype(int)
    
    # Filter by confidence threshold
    mask = scores >= confidence_threshold
    boxes = boxes[mask]
    scores = scores[mask]
    labels = labels[mask]
    
    # Draw predictions on the image
    img_annotated = img.copy()
    for box, score, label in zip(boxes, scores, labels):
        x1, y1, x2, y2 = box.astype(int)
        class_name = class_names[label]
        color = (0, 0, 255)  # Red (BGR)
        thickness = 2
        cv2.rectangle(img_annotated, (x1, y1), (x2, y2), color, thickness)
        label_text = f"{class_name}: {score:.2f}"
        cv2.putText(img_annotated, label_text, (x1, y1-10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, thickness)
    
    # Ensure output directory exists
    os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
    
    # Save annotated image
    cv2.imwrite(output_path, img_annotated)
    print(f"Annotated image saved to {output_path}")
    
    return boxes, scores, labels

# Main execution
if __name__ == "__main__":
    # Image size
    IMG_SIZE = 512
    
    # Input and output paths
    input_image_path = "/kaggle/working/teting_generade_weapon-1/train/images/018743155e71eafad2ad5c55b25703c4_jpg.rf.87d4bb5bf494c67eff7d6ce04074811d.jpg"  # Corrected extension
    output_image_path = "/kaggle/working/annotated_output.jpg"
    
    # Create and load model
    model = create_model(num_classes=num_classes, image_size=IMG_SIZE, architecture='efficientdet_d0')
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model checkpoint not found at {model_path}")
    checkpoint = torch.load(model_path, map_location=device, weights_only=True)
    model.model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    print(f"Loaded model from {model_path}")
    
    # Predict and save
    try:
        boxes, scores, labels = predict_single_image(
            model, 
            input_image_path, 
            output_image_path, 
            confidence_threshold=0.25, 
            image_size=IMG_SIZE
        )
        print("Prediction complete!")
        print(f"Detected {len(boxes)} objects:")
        for box, score, label in zip(boxes, scores, labels):
            print(f"Class: {class_names[label]}, Score: {score:.2f}, Box: {box}")
    except Exception as e:
        print(f"Error during prediction: {e}")


Using device: cuda
Classes: ['Grenade', 'Gun', 'Handgun', 'Knife', 'Person', 'Weapon Holding']
Number of classes: 6
Loaded model from //kaggle/working/efficientdet_weapon_detection.pth
Annotated image saved to /kaggle/working/annotated_output.jpg
Prediction complete!
Detected 0 objects:
